# Load Data

NOTE: before running this section of the code, you will need to add the `Frozone_Data` folder to `data/experiment_results`. This folder should be exactly as provided by Laur in the shared drive.

In [ ]:
import pandas as pd
from pathlib import Path

include_pilot = True

repo_root = Path("__file__").resolve().parent.parent.parent
data_path_414 = repo_root / "data/experiment_results/Frozone_Data/survey2/4-14_survey2_values.csv"
results_data = pd.read_csv(data_path_414, header=1, skiprows=[2])

if include_pilot:
    data_path_pilot = repo_root / "data/experiment_results/Frozone_Data/survey2/pilot_survey2_values.csv"
    results_data_pilot = pd.read_csv(data_path_pilot, header=1, skiprows=[2])
    results_data = pd.concat([results_data, results_data_pilot])
    results_data.reset_index(inplace=True, drop=True)

In [ ]:
results_data.head()

# Data Vis

You may have to install matplotlib and seaborn to run the cells below.

## Violin Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_field_violins(df, category_template):
    """
    Creates violin plots for CName, FName, and HName for a given question category.

    Parameters:
        df: DataFrame loaded from the values CSV (numeric 1-5 responses)
        category_template: Question string with [Field-Name] as placeholder,
                           e.g. "I felt [Field-Name] responded in a way that
                                addressed or countered misrepresentations."
    """
    col_map = {
        "Coolbot": category_template.replace("[Field-Name]", "[Field-CName]"),
        "Frobot": category_template.replace("[Field-Name]", "[Field-FName]"),
        "Hotbot": category_template.replace("[Field-Name]", "[Field-HName]"),
    }

    missing = [col for col in col_map.values() if col not in df.columns]
    if missing:
        raise ValueError(f"Columns not found in DataFrame:\n" + "\n".join(missing))

    plot_df = pd.concat([
        df[col].dropna().rename("Response").to_frame().assign(Field=label)
        for label, col in col_map.items()
    ])

    order = ["Frobot", "Coolbot", "Hotbot"]

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.violinplot(data=plot_df, x="Field", y="Response", inner="box",
                   order=order, ax=ax)

    for i, label in enumerate(order):
        mean_val = plot_df[plot_df["Field"] == label]["Response"].mean()
        ax.scatter(i, mean_val, color="white", edgecolor="black", zorder=5, s=60,
                   label=f"{label} mean: {mean_val:.2f}")

    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(["1 (Strongly Disagree)", "2", "3 (Neutral)", "4", "5 (Strongly Agree)"])
    ax.set_ylim(0.5, 5.5)
    ax.set_title(category_template.replace("[Field-Name]", "each participant"), wrap=True)
    ax.set_xlabel("Chatbot")
    ax.set_ylabel("Response")
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_field_violins(results_data, "I felt [Field-Name] responded in a way that addressed or countered misrepresentations.")


In [ ]:
VALID_VIOLIN_CATEGORIES = [
    "I felt heard and understood by [Field-Name].",
    "I gained a better understanding of [Field-Name]'s perspective.",
    "I feel [Field-Name]'s responses were productive and encouraged frank discussion.",
    "[Field-Name] used toxic or inflammatory language.",
    "I felt [Field-Name] responded in a way that addressed and countered toxic or inflammatory language.",
    "[Field-Name] shared information I believe was factually incorrect.",
    "I felt [Field-Name] responded in a way that addressed and countered information I believe was incorrect.",
    "[Field-Name] made biased comments towards or against groups of people (concerning race, gender, class, sexuality, or something else).",
    "I felt [Field-Name] responded in a way that addressed and countered bias.",
    "[Field-Name]'s reasoning was sometimes flawed.",
    "I felt [Field-Name] responded in a way that addressed flawed reasoning in others' responses.",
    "[Field-Name] misrepresented what I said or what others said.",
    "I felt [Field-Name] responded in a way that addressed or countered misrepresentations.",
]


### Violin Plots across all numerical categories

In [ ]:
for category in VALID_VIOLIN_CATEGORIES:
    plot_field_violins(results_data, category)


## Overall Heatmap

In [ ]:
SHORT_LABELS = [
    "Felt heard and understood",
    "Gained better understanding of bot's perspective",
    "Responses were productive and encouraged frank discussion",
    "Used toxic or inflammatory language",
    "Responded to counter toxic or inflammatory language",
    "Shared factually incorrect information",
    "Responded to counter incorrect information",
    "Made biased comments towards or against groups",
    "Responded to counter bias",
    "Reasoning was sometimes flawed",
    "Addressed flawed reasoning in others' responses",
    "Misrepresented what was said",
    "Addressed or countered misrepresentations",
]

heatmap_data = {}
for label, category in zip(SHORT_LABELS, VALID_VIOLIN_CATEGORIES):
    heatmap_data[label] = {
        "Frobot":  results_data[category.replace("[Field-Name]", "[Field-FName]")].mean(),
        "Coolbot": results_data[category.replace("[Field-Name]", "[Field-CName]")].mean(),
        "Hotbot":  results_data[category.replace("[Field-Name]", "[Field-HName]")].mean(),
    }

heatmap_df = pd.DataFrame(heatmap_data).T[["Frobot", "Coolbot", "Hotbot"]]

fig, ax = plt.subplots(figsize=(7, 9))
sns.heatmap(
    heatmap_df,
    annot=True, fmt=".2f",
    vmin=1, vmax=5,
    cmap="RdYlGn",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Mean Response (1=Strongly Disagree, 5=Strongly Agree)"},
)
ax.set_title("Mean Survey Responses by Chatbot and Category")
ax.set_xlabel("Chatbot")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
NEGATIVE_CATEGORIES = {
    "[Field-Name] used toxic or inflammatory language.",
    "[Field-Name] shared information I believe was factually incorrect.",
    "[Field-Name] made biased comments towards or against groups of people (concerning race, gender, class, sexuality, or something else).",
    "[Field-Name]'s reasoning was sometimes flawed.",
    "[Field-Name] misrepresented what I said or what others said.",
}

heatmap_data_normed = {}
for label, category in zip(SHORT_LABELS, VALID_VIOLIN_CATEGORIES):
    row = {}
    for bot, field in [("Frobot", "FName"), ("Coolbot", "CName"), ("Hotbot", "HName")]:
        col = category.replace("[Field-Name]", f"[Field-{field}]")
        mean = results_data[col].mean()
        if category in NEGATIVE_CATEGORIES:
            mean = 6 - mean  # flip: 1=negative, 5=positive
        row[bot] = mean
    heatmap_data_normed[label] = row

heatmap_df_normed = pd.DataFrame(heatmap_data_normed).T[["Frobot", "Coolbot", "Hotbot"]]

fig, ax = plt.subplots(figsize=(7, 9))
sns.heatmap(
    heatmap_df_normed,
    annot=True, fmt=".2f",
    vmin=1, vmax=5,
    cmap="RdYlGn",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Mean Response (1=Felt Most Negative, 5=Felt Most Positive)"},
)
ax.set_title("Normalized Mean Survey Responses by Chatbot and Category\n(scale normalized: 1=negative, 5=positive)")
ax.set_xlabel("Chatbot")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## Statistical Tests

For each category, a **Friedman test** (non-parametric repeated-measures) checks whether any difference exists across the three bots. Then all **39 pairwise Wilcoxon signed-rank tests** (13 categories × 3 bot pairs) are corrected together using **Benjamini-Hochberg FDR**, directly controlling for the full family of 39 hypotheses.

In [ ]:
import itertools
import numpy as np
from scipy.stats import friedmanchisquare
from statsmodels.stats.multitest import multipletests

BOTS = [("Frobot", "FName"), ("Coolbot", "CName"), ("Hotbot", "HName")]

# --- Friedman omnibus test per category ---
friedman_records = []
for category, label in zip(VALID_VIOLIN_CATEGORIES, SHORT_LABELS):
    cols = [category.replace("[Field-Name]", f"[Field-{field}]") for _, field in BOTS]
    paired = results_data[cols].dropna()
    stat, p = friedmanchisquare(*[paired[c] for c in cols])
    friedman_records.append({"Category": label, "Chi2": stat, "p_raw": p})

friedman_df = pd.DataFrame(friedman_records)
_, p_corr, _, _ = multipletests(friedman_df["p_raw"], method="fdr_bh")
friedman_df["p_corrected (Benjamini-Hochberg)"] = p_corr
friedman_df["significant"] = friedman_df["p_corrected (Benjamini-Hochberg)"] < 0.05

friedman_df.style.format({"Chi2": "{:.3f}", "p_raw": "{:.4f}", "p_corrected (Benjamini-Hochberg)": "{:.4f}"})

In [ ]:
from scipy.stats import wilcoxon

# --- All 39 pairwise Wilcoxon signed-rank tests ---
pairwise_records = []
for category, label in zip(VALID_VIOLIN_CATEGORIES, SHORT_LABELS):
    for (bot1, field1), (bot2, field2) in itertools.combinations(BOTS, 2):
        col1 = category.replace("[Field-Name]", f"[Field-{field1}]")
        col2 = category.replace("[Field-Name]", f"[Field-{field2}]")
        paired = results_data[[col1, col2]].dropna()
        x, y = paired[col1].values, paired[col2].values
        if (x == y).all():
            stat, p = np.nan, 1.0
        else:
            stat, p = wilcoxon(x, y)
        pairwise_records.append({
            "Category": label,
            "Comparison": f"{bot1} vs. {bot2}",
            "W": stat,
            "p_raw": p,
        })

pairwise_df = pd.DataFrame(pairwise_records)

# Benjamini-Hochberg correction across all 39 tests simultaneously
_, p_corr, _, _ = multipletests(pairwise_df["p_raw"].fillna(1.0), method="fdr_bh")
pairwise_df["p_corrected (Benjamini-Hochberg)"] = p_corr
pairwise_df["significant"] = pairwise_df["p_corrected (Benjamini-Hochberg)"] < 0.05

pairwise_df.style.format({"W": "{:.1f}", "p_raw": "{:.4f}", "p_corrected (Benjamini-Hochberg)": "{:.4f}"})

Plots based off of the results of the statistical tests:

In [ ]:
from scipy.stats import rankdata as _rankdata
from matplotlib.patches import Rectangle

def rank_biserial_corr(x, y):
    """Rank-biserial correlation for paired data. Positive = x tends to be larger than y."""
    diff = x - y
    diff = diff[diff != 0]
    n = len(diff)
    if n == 0:
        return 0.0
    ranks = _rankdata(np.abs(diff))
    return (np.sum(ranks[diff > 0]) - np.sum(ranks[diff < 0])) / (n * (n + 1) / 2)

# Compute rank-biserial correlation for each of the 39 pairs
# For negative categories, flip sign so positive always means left bot performed better
rb_vals = []
for category in VALID_VIOLIN_CATEGORIES:
    sign = -1 if category in NEGATIVE_CATEGORIES else 1
    for (bot1, field1), (bot2, field2) in itertools.combinations(BOTS, 2):
        col1 = category.replace("[Field-Name]", f"[Field-{field1}]")
        col2 = category.replace("[Field-Name]", f"[Field-{field2}]")
        paired = results_data[[col1, col2]].dropna()
        rb_vals.append(sign * rank_biserial_corr(paired[col1].values, paired[col2].values))

pairwise_df["rank_biserial"] = rb_vals

# Pivot to category × comparison matrices
effect_matrix = pairwise_df.pivot(index="Category", columns="Comparison", values="rank_biserial").reindex(SHORT_LABELS)
sig_matrix    = pairwise_df.pivot(index="Category", columns="Comparison", values="significant").reindex(SHORT_LABELS)

comparisons = ["Frobot vs. Hotbot", "Coolbot vs. Hotbot", "Frobot vs. Coolbot"]
fig, axes = plt.subplots(1, 3, figsize=(14, 9), sharey=True)

for ax, comp in zip(axes, comparisons):
    col_effect = effect_matrix[[comp]]
    col_sig    = sig_matrix[[comp]]

    annot = col_effect.copy().astype(object)
    for cat in col_effect.index:
        val = col_effect.loc[cat, comp]
        annot.loc[cat, comp] = f"{val:.2f}" if not np.isnan(val) else ""

    sns.heatmap(
        col_effect,
        annot=annot, fmt="",
        center=0, vmin=-1, vmax=1,
        cmap="RdBu",
        linewidths=0.5,
        ax=ax,
        cbar=False,
        yticklabels=True,
    )
    ax.set_title(comp, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")

    # Thick border around significant cells
    for i, cat in enumerate(effect_matrix.index):
        if col_sig.loc[cat, comp]:
            ax.add_patch(Rectangle((0, i), 1, 1, fill=False, edgecolor="black", lw=2.5))

fig.suptitle(
    "Pairwise Effect Sizes (Rank-Biserial Correlation, normalized)\n"
    "Positive = left bot performed better  |  Bold border = significant after Benjamini-Hochberg correction (α = 0.05)",
    fontsize=10,
)

sm = plt.cm.ScalarMappable(cmap="RdBu", norm=plt.Normalize(vmin=-1, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[-1], shrink=0.6, pad=0.05, location="right")
cbar.set_label("Rank-Biserial Correlation\n(positive = left bot performed better)", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Mean difference heatmap (same significance borders from Wilcoxon/Benjamini-Hochberg)
md_vals = []
for category in VALID_VIOLIN_CATEGORIES:
    sign = -1 if category in NEGATIVE_CATEGORIES else 1
    for (bot1, field1), (bot2, field2) in itertools.combinations(BOTS, 2):
        col1 = category.replace("[Field-Name]", f"[Field-{field1}]")
        col2 = category.replace("[Field-Name]", f"[Field-{field2}]")
        paired = results_data[[col1, col2]].dropna()
        md_vals.append(sign * (paired[col1].mean() - paired[col2].mean()))

pairwise_df["mean_diff"] = md_vals

md_matrix = pairwise_df.pivot(index="Category", columns="Comparison", values="mean_diff").reindex(SHORT_LABELS)

comparisons = ["Frobot vs. Hotbot", "Coolbot vs. Hotbot", "Frobot vs. Coolbot"]
fig, axes = plt.subplots(1, 3, figsize=(14, 9), sharey=True)

vabs = max(abs(md_matrix.values.min()), abs(md_matrix.values.max()))

for ax, comp in zip(axes, comparisons):
    col_md  = md_matrix[[comp]]
    col_sig = sig_matrix[[comp]]

    annot = col_md.copy().astype(object)
    for cat in col_md.index:
        val = col_md.loc[cat, comp]
        annot.loc[cat, comp] = f"{val:.2f}" if not np.isnan(val) else ""

    sns.heatmap(
        col_md,
        annot=annot, fmt="",
        center=0, vmin=-vabs, vmax=vabs,
        cmap="RdBu",
        linewidths=0.5,
        ax=ax,
        cbar=False,
        yticklabels=True,
    )
    ax.set_title(comp, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")

    for i, cat in enumerate(md_matrix.index):
        if col_sig.loc[cat, comp]:
            ax.add_patch(Rectangle((0, i), 1, 1, fill=False, edgecolor="black", lw=2.5))

fig.suptitle(
    "Pairwise Mean Differences (normalized)\n"
    "Positive = left bot performed better  |  Bold border = significant after Benjamini-Hochberg correction (α = 0.05)",
    fontsize=10,
)

sm = plt.cm.ScalarMappable(cmap="RdBu", norm=plt.Normalize(vmin=-vabs, vmax=vabs))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[-1], shrink=0.6, pad=0.05, location="right")
cbar.set_label("Mean Difference in Likert Score\n(positive = left bot performed better)", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Adjusted p-value heatmap — color and annotation both show p_corrected
p_matrix = pairwise_df.pivot(index="Category", columns="Comparison", values="p_corrected (Benjamini-Hochberg)").reindex(SHORT_LABELS)
sig_matrix_local = pairwise_df.pivot(index="Category", columns="Comparison", values="significant").reindex(SHORT_LABELS)

comparisons = ["Frobot vs. Hotbot", "Coolbot vs. Hotbot", "Frobot vs. Coolbot"]
fig, axes = plt.subplots(1, 3, figsize=(14, 9), sharey=True)

for ax, comp in zip(axes, comparisons):
    col_p   = p_matrix[[comp]].astype(float)
    col_sig = sig_matrix_local[[comp]]

    annot = col_p.copy().astype(object)
    for cat in col_p.index:
        val = col_p.loc[cat, comp]
        annot.loc[cat, comp] = f"{val:.3f}" if not np.isnan(val) else ""

    sns.heatmap(
        col_p,
        annot=annot, fmt="",
        vmin=0, vmax=1,
        cmap="YlOrRd_r",
        linewidths=0.5,
        ax=ax,
        cbar=False,
        yticklabels=True,
    )
    ax.set_title(comp, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")

    for i, cat in enumerate(p_matrix.index):
        if col_sig.loc[cat, comp]:
            ax.add_patch(Rectangle((0, i), 1, 1, fill=False, edgecolor="black", lw=2.5))

fig.suptitle(
    "Benjamini-Hochberg-Corrected p-values by Category\nBold border = significant (p_corrected < 0.05)",
    fontsize=10,
)

sm = plt.cm.ScalarMappable(cmap="YlOrRd_r", norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[-1], shrink=0.6, pad=0.05, location="right")
cbar.set_label("Benjamini-Hochberg-Corrected p-value\n(darker = more significant)", fontsize=9)
cbar.ax.axhline(0.05, color="black", lw=1.5, linestyle="--")
cbar.ax.text(1.1, 0.05, "α=0.05", va="center", fontsize=8, transform=cbar.ax.get_yaxis_transform())

plt.tight_layout()
plt.show()